# Tabelas de resultados do README

Gera as tabelas da seção 5 do [README](../../README.md) a partir das saídas do processo, no mesmo formato
de SP (`SP/BW/3-Py/readme/tabelas_readme.ipynb`), só com as linhas que valem para RJ e CE:

| Tabela | Seção do README | Origem |
|---|---|---|
| Conta de chegada, valores absolutos e % da receita | 5.1 | pxq (`1-Entrega/pxq/`), aba `Componentes` |

As seções 5.3 (Mercado Receita) e 5.4 (tarifa homologada × realizada) entram quando os checks de CE e RJ
existirem. A linha (g1) de SP (TUSD de Consumo Próprio não faturada por decisão da Empresa) não se aplica.

Saídas em `2-Checks/tabelas_readme/`: `tabelas_readme_<UF>.md` (tabelas prontas para colar no README) e
`tabelas_readme_<UF>.xlsx` (uma aba por tabela, com a explicação de cada uma). O texto de leitura do README
continua manual. Valores monetários em milhões de reais (MR$).

Só o Passo 0 muda: a distribuidora e o período.

In [ ]:
import os
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

# ---------------------------------------------------------------- Passo 0 - parameters
BASE = Path(os.environ.get("PXQ_ROOT", r"C:\python_projects\claude\p_x_q")) / "CE_RJ" / "FAT"

DISTRIBUIDORA = "RJ"                   # "RJ" or "CE"
PERIODO_5_1 = ("2026-10", "2026-10")   # conta de chegada

SAIDA = BASE / "2-Checks" / "tabelas_readme"
SAIDA.mkdir(parents=True, exist_ok=True)
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]


def meses(periodo):
    """['2026-01', ...] for an (start, end) period."""
    return [p.strftime("%Y-%m") for p in pd.period_range(periodo[0], periodo[1], freq="M")]


def rotulo(ref):
    """'2026-03' -> 'mar'."""
    return MESES[int(ref[5:]) - 1]


def br(v, casas=1, sinal=False):
    """Brazilian number format, half-up rounding, true minus sign; '+' on request."""
    if v is None or pd.isna(v):
        return "—"
    t = Decimal(str(float(v))).quantize(Decimal(1).scaleb(-casas), ROUND_HALF_UP)
    if t == 0:
        t = abs(t)
    s = f"{t:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".").replace("-", "−")
    return ("+" + s) if (sinal and t > 0) else s


def tabela_md(df, casas=1, sinal=False, negrito=()):
    """Markdown table (index = first column), numbers in Brazilian format."""
    cab = "| " + " | ".join([df.index.name or ""] + [str(c) for c in df.columns]) + " |"
    sep = "|" + "---|" * (len(df.columns) + 1)
    linhas = [cab, sep]
    for nome, r in df.iterrows():
        cel = [br(v, casas, sinal) if not isinstance(v, str) else v for v in r]
        if nome in negrito:
            # bold the label and the numbers, not the text columns (unit)
            nome = f"**{nome}**"
            cel = [f"**{c}**" if (c != "—" and not isinstance(v, str)) else c for c, v in zip(cel, r)]
        linhas.append("| " + " | ".join([str(nome)] + cel) + " |")
    return "\n".join(linhas)


TABELAS = []   # (sheet name, README section, explanation, DataFrame, md text)
print("distribuidora:", DISTRIBUIDORA, "| periodo 5.1:", PERIODO_5_1)

## 5.1 — Do valor pela tarifa ao valor faturado

Conta de chegada, por mês, sobre a receita tarifável: (a) receita = (b) bloco 1 + (d) Tarifa Social +
(e) APE + (f) fonte incentivada + (g) 2ª passada + (h) resíduo. A linha (i) soma os valores absolutos da
2ª passada de todos os grupos: mostra se o resultado líquido pequeno esconde ajustes grandes de sinais
opostos.

In [ ]:
LINHAS_5_1 = {
    "a": "(a) Receita faturada (base tarifável)", "b": "(b) Bloco 1 (tarifa cheia)",
    "c": "(c) Diferença a explicar, (b) − (a)", "d": "(d) Tarifa Social (faixa gratuita)",
    "e": "(e) Isenção APE", "f": "(f) Fonte incentivada (1ª passada)", "g": "(g) 2ª passada, líquida",
    "h": "(h) Resíduo sem item tarifário", "i": "(i) 2ª passada, bruta",
}
valores = {}
for ref in meses(PERIODO_5_1):
    c = pd.read_excel(BASE / "1-Entrega" / "pxq" / f"pxq_{DISTRIBUIDORA}_{ref}.xlsx", sheet_name="Componentes")
    c = c[c["Tarifável?"].eq("SIM")]
    col = lambda n: c[n].fillna(0.0)
    a, b = col("RECEITA DE FORNECIMENTO").sum(), col("VALOR CALCULADO (bloco 1)").sum()
    d, e = col("Ajuste Tarifa Social (R$)").sum(), col("Ajuste regime APE (R$)").sum()
    f, g = col("Ajuste 1ª passada (R$)").sum(), col("Ajuste 2ª passada (R$)")
    valores[rotulo(ref)] = {"a": a, "b": b, "c": b - a, "d": d, "e": e, "f": f, "g": g.sum(),
                            "h": a - b - d - e - f - g.sum(),
                            "i": g.abs().sum()}
t = pd.DataFrame(valores) / 1e6

t51a = t.rename(index=LINHAS_5_1)
t51a.insert(0, "Unidade", "R$ milhões")
t51a.index.name = "Item"
pct = (t.drop(index=["a", "b"]) / t.loc["a"] * 100)
pct.index = [LINHAS_5_1[k] for k in pct.index]
t51b = pct.copy()
t51b.insert(0, "Unidade", "% de (a)")
t51b.index.name = "Item"

md_a = tabela_md(t51a, 1)
md_b = tabela_md(t51b.assign(**{k: t51b[k] for k in t51b.columns}), 2, sinal=False)
# sign on the net lines, as in the README
md_b = "\n".join(l if not (l.startswith("| (g)")) else
                  "| " + " | ".join([p.strip() if (i < 2 or p.strip().startswith(("−", "0,00"))) else "+" + p.strip()
                                    for i, p in enumerate(l.strip("|").split("|"))]) + " |"
                  for l in md_b.split("\n"))
TABELAS.append(("5.1 valores", "5.1", "Conta de chegada em R$ milhões: o que a tarifa explica sozinha (bloco 1) "
                "e o que explica a diferença até a receita. É a tabela \"Valores absolutos\" da seção 5.1 do README.",
                t51a, md_a))
TABELAS.append(("5.1 percentual", "5.1", "As mesmas linhas em % da receita faturada (a). É a tabela \"Em relação à "
                "receita faturada\" da seção 5.1 do README.", t51b, md_b))
print(md_a); print(); print(md_b)

## Gravar `.md` e `.xlsx`

In [ ]:
# markdown: one block per table, ready to paste in the README
partes = [f"# Tabelas de resultados do README — {DISTRIBUIDORA}", "",
          "Gerado por `3-Py/readme/tabelas_readme.ipynb`. Valores em R$ milhões ou %, conforme a coluna Unidade.", ""]
for aba, secao, texto, _, txt in TABELAS:
    partes += [f"## {aba} (README, seção {secao})", "", texto, "", txt, ""]
(SAIDA / f"tabelas_readme_{DISTRIBUIDORA}.md").write_text("\n".join(partes), encoding="utf-8")

# Excel: Leia-me + one sheet per table (explanation on top, table below)
arq_x = SAIDA / f"tabelas_readme_{DISTRIBUIDORA}.xlsx"
with pd.ExcelWriter(arq_x, engine="openpyxl") as w:
    pd.DataFrame([(a, f"Seção {s}", t) for a, s, t, _, _ in TABELAS],
                 columns=["Aba", "README", "O que mostra"]).to_excel(w, sheet_name="Leia-me", index=False)
    for aba, secao, texto, df, _ in TABELAS:
        df.reset_index().to_excel(w, sheet_name=aba, index=False, startrow=3)
        ws = w.book[aba]
        ws["A1"] = f"README, seção {secao}"
        ws["A1"].font = Font(bold=True, size=11)
        ws["A2"] = texto
        ws["A2"].alignment = Alignment(wrap_text=True, vertical="top")
        ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=min(df.shape[1] + 1, 12))
        ws.row_dimensions[2].height = 60
        for cel in ws[4]:
            cel.font = Font(bold=True, color="FFFFFF")
            cel.fill = PatternFill("solid", fgColor="1F4E79")
            cel.alignment = Alignment(horizontal="center", wrap_text=True)
        ws.column_dimensions["A"].width = 58
        for j in range(2, ws.max_column + 1):
            ws.column_dimensions[get_column_letter(j)].width = 13
            for i in range(5, ws.max_row + 1):
                ws.cell(i, j).number_format = "#,##0.00" if "%" in str(df.iloc[0, 0]) else "#,##0.0"
    lm = w.book["Leia-me"]
    lm.column_dimensions["A"].width = 18
    lm.column_dimensions["B"].width = 12
    lm.column_dimensions["C"].width = 110
    for row in lm.iter_rows(min_row=2):
        row[2].alignment = Alignment(wrap_text=True, vertical="top")
print("gravado:", SAIDA / f"tabelas_readme_{DISTRIBUIDORA}.md")
print("gravado:", arq_x)